In [1]:
import torch
from test_attn_order_eval import AttnOrderEval   # signal-agnostic ranking eval over a [T, L] per-step score + order


class MarginOrderEval(AttnOrderEval):

    def __init__(self, margin, order):                              # margin [T, L] (already preprocessed, p1 - p2), order [T] long
        assert margin.dim() == 2, "margin.dim(): {} == 2 false".format(margin.dim())

        super().__init__(margin, order)                             # reuse geometry + recall_at_h / pr_auc / ndcg_at_h
    # end

    def get_margin(self):
        return self.attn   # the base stores the ranking score here (generic [T, L] score slot)
    # end
# end



In [ ]:
if __name__ == "__main__":
    torch.manual_seed(0)
    T = L = 64

    folder_base = 'stats/0'
    pos_base = 32

    unmask = torch.load(f'{folder_base}/unmask_{pos_base}_{pos_base + L}.pt')
    unmask = unmask.squeeze(-1) - pos_base
    order = unmask


    step_of = torch.full((L,), -1, dtype=torch.long)
    step_of[order] = torch.arange(T)
    gap = step_of.view(1, L) - torch.arange(T).view(T, 1)          # [T, L]

    margin_perfect = torch.where(gap > 0, 1.0 / gap.clamp_min(1).double(), torch.zeros(1, dtype=torch.float64))   # larger margin = sooner
    margin_random = torch.rand(T, L, dtype=torch.float64)
    margin = torch.load(f'{folder_base}/margin_{pos_base}_{pos_base + L}.pt')


    def summ(x):
        return "{:.3f} (n={})".format(x.nanmean().item(), int((~x.isnan()).sum()))
    # end

    for name, margin in [("perfect", margin_perfect), ("random ", margin_random), ("margin", margin)]:
        ev = MarginOrderEval(margin, order)
        print("{}  recall@5={}  pr_auc@5={}  ndcg@10={}".format(
            name, summ(ev.recall_at_h(5)), summ(ev.pr_auc(5)), summ(ev.ndcg_at_h(10))))
    # end
# end

torch.Size([64, 64]) torch.Size([64, 64])
perfect  recall@5=1.000 (n=59)  pr_auc@5=1.000 (n=58)  ndcg@10=1.000 (n=62)
random   recall@5=0.275 (n=59)  pr_auc@5=0.319 (n=58)  ndcg@10=0.412 (n=62)
margin  recall@5=0.600 (n=59)  pr_auc@5=0.687 (n=58)  ndcg@10=0.759 (n=62)
